# Space & Astronomy RAG Assistant

Retrieval-Augmented Generation system over astronomy/space PDFs, using HuggingFace embeddings, FAISS, and a Groq-hosted LLM.

**Structure of this notebook:**
1. Setup & installs
2. OCR + chunking
3. FAISS vector index
4. Groq LLM + RAG chain
5. Quick test
6. **Gradio interface** (run this section for a demo/CV screenshot)
7. **RAGAS evaluation** (optional — run in a fresh runtime session, separately from the interface section, see note below)

## 1. Setup & Installs

In [1]:
!apt-get -q install -y poppler-utils tesseract-ocr
!pip install -q langchain langchain-community langchain-classic langchain-groq \
    langchain-huggingface sentence-transformers faiss-cpu pypdf pytesseract pdf2image gradio

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.13 [186 kB]
Fetched 186 kB in 1s (327 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.2 M

In [11]:
# Set your Groq API key (prompted securely, never hardcoded in the notebook)
import os
from getpass import getpass

key = getpass("Enter your Groq API key: ").strip()

if not key:
    raise ValueError("No API key entered — the input was empty. Re-run this cell and paste your key again.")

os.environ["GROQ_API_KEY"] = key
print(f"Key set (length: {len(key)} characters)")  # sanity check without printing the key itself


Enter your Groq API key: ··········
Key set (length: 56 characters)


## 2. Load PDF with OCR + Chunking

In [3]:
# Upload your PDF file. In Google Colab this opens a file picker.
try:
    from google.colab import files
    uploaded = files.upload()
    PDF_PATH = "/content/" + list(uploaded.keys())[0]
except ImportError:
    # Not running in Colab — set the path manually instead.
    PDF_PATH = "/content/Space_Astronomy_RAG_Assistant (4).ipynb"

import os
assert os.path.exists(PDF_PATH), f"File not found at {PDF_PATH} — check the upload succeeded."
print(f"Using file: {PDF_PATH}")

Saving hubblefocusouramazingsolarsystem.pdf to hubblefocusouramazingsolarsystem.pdf
Using file: /content/hubblefocusouramazingsolarsystem.pdf


In [4]:
from pdf2image import convert_from_path
import pytesseract
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_pdf_with_ocr(file_path):
    """Extract text from an image-based PDF using OCR (Tesseract)."""
    images = convert_from_path(file_path)
    documents = []

    for i, img in enumerate(images):
        text = pytesseract.image_to_string(img, lang="eng")
        if text.strip():
            documents.append(
                Document(
                    page_content=text,
                    metadata={"source": file_path, "page": i + 1}
                )
            )
    return documents


docs = load_pdf_with_ocr(PDF_PATH)
print(f"Extracted text from {len(docs)} pages")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)
print(f"Loaded {len(docs)} pages -> split into {len(chunks)} chunks")

Extracted text from 66 pages
Loaded 66 pages -> split into 262 chunks


## 3. Embeddings + FAISS Index

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector = FAISS.from_documents(documents=chunks, embedding=embeddings_model)
vector.save_local("/content/faiss_index")

print("FAISS index built and saved successfully!")

/tmp/ipykernel_1028/3004785200.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built and saved successfully!


## 4. Groq LLM + RAG Chain

In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

# Quick sanity check
response = llm.invoke("What is the solar system?")
print(response.content)

The **Solar System** is the collection of celestial bodies that are bound together by the Sun’s gravity. It includes:

1. **The Sun** – a middle‑size (G‑type) star that contains more than 99 % of the system’s mass and provides the light and heat that make life possible on Earth.  
2. **Eight major planets** – Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune, which orbit the Sun in roughly the same plane (the ecliptic).  
3. **Dwarf planets** – bodies like Pluto, Eris, Haumea, Makemake, and Ceres that are massive enough to be round but have not cleared their orbital neighborhoods.  
4. **Moons (natural satellites)** – over 200 known moons orbiting the planets and dwarf planets (e.g., Earth’s Moon, Jupiter’s Ganymede, Saturn’s Titan).  
5. **Small Solar‑System bodies** – asteroids (mostly in the asteroid belt between Mars and Jupiter), comets (icy bodies that develop tails when near the Sun), meteoroids, and dust particles.  
6. **The Kuiper Belt and Oort Cloud** – dista

In [13]:
def get_retriever(k=4):
    return vector.as_retriever(search_kwargs={"k": k})


retriever = get_retriever(k=4)

from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

prompt_template = """You are a space and astronomy expert assistant.

Answer the question using ONLY the context provided below.

If the context does not contain enough information, say you don't have enough information — do not hallucinate.

For multi-part questions, address each part separately using the retrieved context.

Context:
{context}

Question:
{question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

print("RAG chain created successfully!")

RAG chain created successfully!


## 5. Quick Test

In [14]:
query = "What kind of weather patterns exist on the different planets?"
result = qa.invoke({"query": query})

print("Answer:", result["result"])
print("\nSources used:")
for doc in result["source_documents"]:
    print("-", doc.metadata.get("source", "unknown"), "- page", doc.metadata.get("page"))

Answer: **Saturn**  
- Shows **banded cloud patterns** with **alternating wind directions** at different latitudes.  
- The bands are covered by a **deep, high‑altitude methane haze**, giving the planet a more subdued colour than Jupiter.  
- Because its **axial tilt is similar to Earth’s**, Saturn experiences **seasonal changes** that have been observed, although its **internal heat** keeps the overall temperature fairly constant across the planet and throughout the year.

**Jupiter**  
- Wrapped in clouds of **ammonia crystals** that create a **striped (banded) appearance** of yellow, brown and white hues.  
- The bands are divided into **zones** (lighter, high‑altitude regions where air **rises**) and **belts** (darker, low‑altitude regions where air **falls**).  
- With only a **3‑degree axial tilt**, Jupiter has **practically no seasonal changes**; its weather is driven mainly by **heat rising from its interior** (several times hotter than the Sun’s surface) plus a weak solar inpu

## 6. Gradio Interface

Run this section for the interactive demo (useful for a CV/portfolio screenshot or link).

**Note:** run this section as-is (right after the cells above). Do NOT run the RAGAS evaluation section (below) before this one in the same session — RAGAS internally patches Python's `asyncio`, which breaks Gradio's server. If you want to run both, restart the runtime between them, or run this section first.

In [15]:
import gradio as gr


def answer_question(question, top_k):
    if not question.strip():
        return "Please enter a question.", ""

    qa.retriever = get_retriever(k=int(top_k))

    try:
        result = qa.invoke({"query": question})
    except Exception as e:
        return f"An error occurred: {e}", ""

    answer = result["result"]
    sources = "\n".join(
        f"- Page {doc.metadata.get('page', '?')}"
        for doc in result["source_documents"]
    )
    return answer, sources


with gr.Blocks(title="Space & Astronomy RAG Assistant") as demo:
    gr.Markdown("# 🚀 Space & Astronomy Q&A Assistant")
    gr.Markdown(
        "Ask questions about planetary science, space missions, and astrophysics — "
        "answers are grounded strictly in the retrieved document."
    )

    with gr.Row():
        with gr.Column(scale=3):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g. What causes Jupiter's striped appearance?",
                lines=2
            )
            top_k_slider = gr.Slider(
                minimum=1, maximum=10, value=4, step=1,
                label="Number of retrieved passages (top-k)"
            )
            submit_btn = gr.Button("Ask", variant="primary")

        with gr.Column(scale=2):
            answer_output = gr.Textbox(label="Answer", lines=8)
            sources_output = gr.Textbox(label="Source Pages", lines=4)

    submit_btn.click(
        fn=answer_question,
        inputs=[question_input, top_k_slider],
        outputs=[answer_output, sources_output]
    )

    gr.Examples(
        examples=[
            ["What kind of weather patterns exist on Mars?", 4],
            ["What causes Jupiter's striped cloud appearance?", 4],
            ["When was the Hubble Space Telescope launched?", 3],
        ],
        inputs=[question_input, top_k_slider]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fbfedbf1bd6d933992.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fbfedbf1bd6d933992.gradio.live


---
## 7. Evaluation (RAGAS) — Optional

**Run this section in a fresh/restarted runtime**, using only cells 1-5 above (setup through the RAG chain) — skip the Gradio section — then run the cells below. This avoids the `asyncio` conflict between RAGAS and Gradio's server.

### 7.1 Ground truth test set

Questions and answers taken directly from the document text (not model-generated), used as reference for scoring.

In [16]:
test_questions = [
    {
        "question": "When was the Hubble Space Telescope launched, and how many observations has it made?",
        "ground_truth": "Hubble was launched in 1990 and has made more than one million observations since then."
    },
    {
        "question": "What is the diameter of Hubble's primary mirror?",
        "ground_truth": "Hubble's primary mirror is 94.5 inches in diameter, and it is the smoothest optical mirror ever polished."
    },
    {
        "question": "How much better can Hubble resolve astronomical objects compared to ground-based telescopes?",
        "ground_truth": "Hubble can resolve astronomical objects ten to twenty times better than typically possible with large ground-based telescopes."
    },
    {
        "question": "Who was the first person to observe the planets with a telescope, and what did he discover about Saturn?",
        "ground_truth": "Galileo Galilei observed the planets with the newly invented telescope in the early 1600s and found that Saturn had 'appendages,' which were later resolved as rings."
    },
    {
        "question": "What spacecraft made the first flyby of Venus and Mars?",
        "ground_truth": "Mariner 2 made the first flyby of Venus in 1962, and Mariner 4 made the first flyby of Mars two years later."
    },
    {
        "question": "What event did Hubble famously photograph involving Jupiter?",
        "ground_truth": "Hubble photographed the collision of Comet Shoemaker-Levy 9 with Jupiter, chronicling the sequential impacts of the comet's fragments on the planet."
    },
    {
        "question": "How long does Mars take to complete one orbit around the Sun, and how does this affect its seasons?",
        "ground_truth": "Mars takes 687 days to complete one orbit, making its seasons about twice as long as Earth's."
    },
    {
        "question": "What is the composition of the Martian atmosphere and how does its pressure compare to Earth's?",
        "ground_truth": "The Martian atmosphere is 95 percent carbon dioxide, and its average pressure is roughly 135 times lower than Earth's atmospheric pressure."
    },
    {
        "question": "What causes Jupiter's striped cloud appearance?",
        "ground_truth": "Jupiter's clouds of ammonia crystals form a striped appearance because they are arranged into bands at different latitudes, produced by air flowing in different directions; lighter zones are high-altitude rising air, and darker belts are low-altitude falling air."
    },
    {
        "question": "What drives Jupiter's weather given its minimal axial tilt?",
        "ground_truth": "With an axial tilt of just 3 degrees, Jupiter has practically no seasonal changes; instead its weather is driven by heat bubbling up from its interior combined with a small amount of solar radiation."
    },
]

print(f"Loaded {len(test_questions)} ground-truth questions")

Loaded 10 ground-truth questions


In [17]:
results_for_eval = []
for item in test_questions:
    result = qa.invoke({"query": item["question"]})
    results_for_eval.append({
        "question": item["question"],
        "answer": result["result"],
        "contexts": [doc.page_content for doc in result["source_documents"]],
        "ground_truth": item["ground_truth"]
    })

print(f"Collected {len(results_for_eval)} results for evaluation")

Collected 10 results for evaluation


### 7.2 Install and patch RAGAS

`ragas` hard-imports a class that was removed from newer `langchain-community` versions. The stub below satisfies that import without needing VertexAI (which this project doesn't use).

In [18]:
!pip install -q "ragas==0.4.3" datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 20.4 MB/s eta 0:00:00


In [19]:
import sys
import types

# ragas tries to import ChatVertexAI from a path removed in current langchain-community.
# We don't use VertexAI (we use Groq), so a stub is safe here.
stub = types.ModuleType('langchain_community.chat_models.vertexai')


class ChatVertexAI:
    def __init__(self, *args, **kwargs):
        raise NotImplementedError('VertexAI is not used in this project.')


stub.ChatVertexAI = ChatVertexAI
sys.modules['langchain_community.chat_models.vertexai'] = stub

import ragas
print("ragas version:", ragas.__version__)

ragas version: 0.4.3


### 7.3 Run evaluation

`answer_relevancy` is set to `strictness=1` because Groq's API only supports `n=1` (single completion per request), and the default strictness of 3 requests 3 completions at once. `max_workers=1` keeps requests sequential to stay under Groq's tokens-per-minute limit — this is slower but avoids failed/skipped evaluations.

In [20]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall
from ragas.metrics import AnswerRelevancy
from ragas.run_config import RunConfig

answer_relevancy_fixed = AnswerRelevancy(strictness=1)

run_config = RunConfig(
    max_workers=1,
    timeout=180,
    max_retries=5,
    max_wait=90
)

eval_dataset = Dataset.from_list(results_for_eval)

scores = evaluate(
    eval_dataset,
    metrics=[faithfulness, answer_relevancy_fixed, context_precision, context_recall],
    llm=llm,
    embeddings=embeddings_model,
    run_config=run_config
)

print(scores)

/tmp/ipykernel_1028/3932061323.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, context_recall
/tmp/ipykernel_1028/3932061323.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision, context_recall
/tmp/ipykernel_1028/3932061323.py:3: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import faithfulness, context_precision, context_recall
/t

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'faithfulness': 0.7471, 'answer_relevancy': 0.8375, 'context_precision': 0.9000, 'context_recall': 0.9000}
